# LLM: pretraining и SFT

Предсказание следующего токена и обучение только на ответе.

> Все примеры этого урока работают на CPU, не скачивают модели и используют искусственные данные. Они объясняют механизм; качество настоящего обучения проверяется отдельно.

## Что важно понять

Causal LM предсказывает следующий токен по предыдущим. В pretraining loss обычно считается на всех непустых токенах; в response-only SFT — на ответе ассистента. Сдвиг logits и labels должен быть сделан ровно один раз.

Ниже небольшая модель с causal attention, а не готовая языковая модель. Данные синтетические. В Training Studio используйте свой train/validation, chat template модели и сначала короткий проверочный запуск.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Запустите эксперимент

Запускайте ячейки по порядку. Код можно менять; Shift + Enter выполняет текущую ячейку.

In [ ]:
class TinyLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(16, 24)
        self.position = nn.Embedding(8, 24)
        self.block = nn.TransformerEncoderLayer(24, 4, 48, dropout=0, batch_first=True)
        self.head = nn.Linear(24, 16)
    def forward(self, ids):
        length = ids.shape[1]
        h = self.embedding(ids) + self.position(torch.arange(length))
        causal = torch.triu(torch.ones(length, length, dtype=torch.bool), diagonal=1)
        return self.head(self.block(h, src_mask=causal))
model = TinyLM(); optimizer = torch.optim.AdamW(model.parameters(), lr=0.02)
batch = torch.tensor([[1, 2, 3, 4, 5, 6, 7, 8]]).repeat(8, 1)
labels = batch[:, 1:].clone(); labels[:, :3] = -100
losses = []
for step in range(35):
    optimizer.zero_grad(set_to_none=True)
    loss = F.cross_entropy(model(batch[:, :-1]).flatten(0, 1), labels.flatten())
    loss.backward(); optimizer.step(); losses.append(loss.item())
print('supervised tokens per sample:', (labels[0] != -100).sum().item())
plt.plot(losses, color='#d73535'); plt.ylabel('response-only CE'); plt.show()

## Попробуйте сами

Уберите маску prompt и сравните loss. Меняйте длину prompt, сохраняя корректное соответствие сдвинутым labels. Почему memorization одного примера не означает полезный SFT?

Сохраните копию ноутбука и запишите, что изменили, чего ожидали и что измерили.

## Продолжить в студии

Кнопка «Открыть раздел студии» над уроком ведёт к связанным инструментам. Учебный код сам не запускает большие тренировки. Перед реальным запуском проверьте данные, ресурсы и поддержку метода у выбранной модели.

## Источники

- [Attention Is All You Need](https://arxiv.org/abs/1706.03762)